# ⚡ GridGuard — Automated Transmission Line Inspection

**GeoVIDINT Hackathon 2026 · Track 2 — Energy Infrastructure Monitoring**

This notebook does ONE thing: it watches drone footage of power lines and automatically finds problems (damaged insulators, trees too close, rusty towers).

### How it works (simple version)
```
Your video → Upload to S3 → Marengo watches it → You search with English → Pegasus describes problems → Report
```

### What each part does
- **S3** = Cloud storage (like Google Drive). Your video lives here.
- **Marengo** = AI that understands video. You ask it questions in English, it finds the right moments.
- **Pegasus** = AI that describes video. It watches a clip and writes what it sees.
- **This notebook** = Your code that connects everything together.

---
## STEP 0: Setup (run this first, don't change anything)

In [ ]:
# Install what we need (this takes ~30 seconds)
%pip install -Uq boto3 pandas plotly numpy

In [ ]:
# Import all the tools we'll use
import boto3
import json
import time
import uuid
import os
import pandas as pd
import numpy as np
from IPython.display import HTML, display, clear_output

# Connect to AWS (this uses the credentials that are already set up in SageMaker)
session = boto3.Session()
AWS_REGION = session.region_name or 'us-east-1'
sts = session.client('sts')
aws_account_id = sts.get_caller_identity()['Account']

# These are the two AI clients we'll use
bedrock_client = session.client('bedrock-runtime')  # For calling Marengo and Pegasus
s3_client = session.client('s3')                     # For uploading/downloading files

print(f'✅ Connected to AWS')
print(f'   Region: {AWS_REGION}')
print(f'   Account: {aws_account_id}')

In [ ]:
# ════════════════════════════════════════════════════════════
# YOUR BUCKET NAME — this is the ONLY thing you change here
# ════════════════════════════════════════════════════════════
S3_BUCKET_NAME = "twelvelabs-bedrock-workshop-workshopbucket-hrd36ffdc1fy"
# ════════════════════════════════════════════════════════════

# Model IDs (don't change these)
MARENGO_MODEL_ID = 'us.twelvelabs.marengo-embed-3-0-v1:0'   # The video search AI
PEGASUS_MODEL_ID = 'us.twelvelabs.pegasus-1-2-v1:0'         # The video description AI

# Verify the bucket exists
try:
    s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
    print(f'✅ S3 bucket verified: {S3_BUCKET_NAME}')
except Exception as e:
    print(f'❌ Bucket error: {e}')
    print('   Check your bucket name above')

---
## STEP 1: Upload your video to S3

**Before running this cell:**
1. Download your transmission line inspection video
2. In SageMaker JupyterLab, click the upload button (⬆️ icon) in the left file browser
3. Upload the video file — it will appear in your file list
4. Change the filename below to match your video file name

In [ ]:
# ════════════════════════════════════════════════════════════
# CHANGE THIS to your video filename
# ════════════════════════════════════════════════════════════
VIDEO_FILENAME = "inspection_footage.mp4"  # <-- change this to your video's filename
# ════════════════════════════════════════════════════════════

# Upload to S3
video_s3_key = f"videos/{VIDEO_FILENAME}"
video_s3_uri = f"s3://{S3_BUCKET_NAME}/{video_s3_key}"

print(f'Uploading {VIDEO_FILENAME} to cloud storage...')
print(f'This may take a few minutes depending on file size.')

s3_client.upload_file(VIDEO_FILENAME, S3_BUCKET_NAME, video_s3_key)

print(f'\n✅ Video uploaded!')
print(f'   Location: {video_s3_uri}')

---
## STEP 2: Marengo watches your video and creates searchable segments

This is an async (background) task. Marengo breaks your video into ~5-10 second segments and creates a mathematical "fingerprint" for each one. This takes **2-5 minutes** depending on video length.

**You don't need to change anything here. Just run the cells and wait.**

In [ ]:
def create_video_embeddings(video_s3_uri):
    """
    Send your video to Marengo. It watches the whole thing
    and creates a 'fingerprint' (512 numbers) for each segment.
    These fingerprints let us search the video with text later.
    """
    unique_id = str(uuid.uuid4())
    output_prefix = f'embeddings/videos/{unique_id}'
    
    # Tell Marengo to start processing the video
    response = bedrock_client.start_async_invoke(
        modelId=MARENGO_MODEL_ID,
        modelInput={
            'inputType': 'video',
            'video': {
                'mediaSource': {
                    's3Location': {
                        'uri': video_s3_uri,
                        'bucketOwner': aws_account_id
                    }
                },
                'embeddingOption': ['visual'],
                'embeddingScope': ['clip']
            }
        },
        outputDataConfig={
            's3OutputDataConfig': {
                's3Uri': f's3://{S3_BUCKET_NAME}/{output_prefix}'
            }
        }
    )
    
    invocation_arn = response['invocationArn']
    print(f'Marengo started processing your video...')
    print(f'Task ID: {invocation_arn}')
    
    # Wait for it to finish (polls every 5 seconds)
    status = None
    while status not in ['Completed', 'Failed', 'Expired']:
        resp = bedrock_client.get_async_invoke(invocationArn=invocation_arn)
        status = resp['status']
        clear_output(wait=True)
        print(f'⏳ Marengo is watching your video... Status: {status}')
        if status not in ['Completed', 'Failed', 'Expired']:
            time.sleep(5)
    
    if status != 'Completed':
        raise Exception(f'Marengo failed: {status}')
    
    # Download the results from S3
    response = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=output_prefix)
    for obj in response.get('Contents', []):
        if obj['Key'].endswith('output.json'):
            result = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=obj['Key'])
            content = result['Body'].read().decode('utf-8')
            data = json.loads(content).get('data', [])
            return data
    
    raise Exception('No output found — Marengo may not have processed the video correctly')

In [ ]:
# Run Marengo on your video (wait 2-5 minutes)
print(f'Sending video to Marengo: {video_s3_uri}')
print(f'This takes 2-5 minutes. Go label your ground truth while you wait!\n')

video_segments = create_video_embeddings(video_s3_uri)

print(f'\n✅ Marengo finished!')
print(f'   Created {len(video_segments)} searchable segments')
print(f'   Each segment is ~{video_segments[0]["endSec"] - video_segments[0]["startSec"]:.0f} seconds long')
print(f'\nFirst 5 segments:')
for seg in video_segments[:5]:
    print(f'   {seg["startSec"]:.1f}s - {seg["endSec"]:.1f}s')

---
## STEP 3: Search the video for anomalies

Now the magic happens. We type descriptions of problems in plain English, and Marengo finds the video moments that match.

**How this works behind the scenes:**
1. Your text query ("damaged insulator") gets turned into a fingerprint (512 numbers)
2. We compare that fingerprint against every video segment's fingerprint
3. The most similar segments are our detections

We're skipping S3 Vectors and doing the comparison right here in memory — faster and simpler.

In [ ]:
# Build the in-memory search index from our video segments
# (This replaces S3 Vectors — same math, no extra AWS service needed)

segment_embeddings = np.array([seg['embedding'] for seg in video_segments])  # Matrix of all fingerprints
segment_times = [(seg['startSec'], seg['endSec']) for seg in video_segments]  # Time ranges

print(f'✅ Search index ready: {segment_embeddings.shape[0]} segments × {segment_embeddings.shape[1]} dimensions')


def search_video(query_text, top_k=3):
    """
    Search the video using a text description.
    Returns the top_k most similar video segments.
    """
    # Turn the text into a fingerprint using Marengo
    model_input = {
        'inputType': 'text',
        'text': {'inputText': query_text}
    }
    response = bedrock_client.invoke_model(
        modelId=MARENGO_MODEL_ID,
        body=json.dumps(model_input)
    )
    query_data = json.loads(response['body'].read().decode('utf-8'))['data']
    query_embedding = np.array(query_data[0]['embedding'])
    
    # Compare against all video segments (cosine similarity)
    # Higher score = more similar to the query
    similarities = np.dot(segment_embeddings, query_embedding) / (
        np.linalg.norm(segment_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    
    # Get the top matches
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'score': float(similarities[idx]),
            'start_time': segment_times[idx][0],
            'end_time': segment_times[idx][1],
            'segment_index': int(idx),
        })
    return results

In [ ]:
# ═══════════════════════════════════════════════════════
# DETECTION QUERIES — search for 3 types of anomalies
# ═══════════════════════════════════════════════════════

DETECTION_QUERIES = {
    'insulator_damage': [
        'damaged insulator on transmission tower',
        'cracked or broken insulator disc on power line',
        'insulator with burn marks or flashover damage',
        'contaminated dirty insulator string on tower',
    ],
    'vegetation_encroachment': [
        'trees growing very close to power lines',
        'vegetation encroaching on transmission line corridor',
        'branches near high voltage conductors',
        'overgrown trees in power line right of way',
    ],
    'structural_corrosion': [
        'rust or corrosion on transmission tower steel',
        'corroded metal structure on power line tower',
        'paint deterioration on tower structure',
        'visible metal degradation on lattice tower',
    ],
}

# Run ALL detection queries and collect results
all_findings = []

for anomaly_type, queries in DETECTION_QUERIES.items():
    print(f'\n══ Searching for: {anomaly_type.replace("_", " ").upper()} ══')
    for query in queries:
        print(f'  Query: "{query}"')
        results = search_video(query, top_k=3)
        for r in results:
            finding = {
                'anomaly_type': anomaly_type,
                'query': query,
                'score': r['score'],
                'start_time': r['start_time'],
                'end_time': r['end_time'],
            }
            all_findings.append(finding)
            print(f'    ✓ Found at {r["start_time"]:.1f}s - {r["end_time"]:.1f}s (similarity: {r["score"]:.3f})')

print(f'\n══ Total raw detections: {len(all_findings)} ══')

In [ ]:
# Remove duplicate detections (same time range found by multiple queries)
def deduplicate(findings, overlap_sec=3.0):
    if not findings:
        return []
    findings.sort(key=lambda f: (f['start_time'], -f['score']))
    kept = [findings[0]]
    for curr in findings[1:]:
        prev = kept[-1]
        overlap = min(prev['end_time'], curr['end_time']) - max(prev['start_time'], curr['start_time'])
        if overlap > overlap_sec:
            if curr['score'] > prev['score']:
                kept[-1] = curr  # Replace with higher-scoring detection
        else:
            kept.append(curr)
    return kept

findings = deduplicate(all_findings)
print(f'After removing duplicates: {len(findings)} unique findings\n')
for i, f in enumerate(findings):
    print(f'  #{i+1} {f["anomaly_type"]:25s} | {f["start_time"]:6.1f}s - {f["end_time"]:6.1f}s | score: {f["score"]:.3f}')

---
## STEP 4: Pegasus describes each finding

For each anomaly Marengo found, we ask Pegasus: "What's wrong with the infrastructure in this part of the video?"

Pegasus watches the video and writes a structured inspection report with:
- What type of asset it is
- What the problem looks like
- How severe it is
- What action to take

We use **structured output** (JSON schema) so Pegasus gives us consistent, parseable data.

Then we apply **NERC FAC-003** severity scoring — real regulatory thresholds, not made-up scales.

In [ ]:
# NERC-based severity framework
# These are based on real regulations, not arbitrary scales
# FAC-003-4: Transmission Vegetation Management
# FAC-501-3: Facility Ratings (structural integrity)
SEVERITY_FRAMEWORK = {
    'critical': {
        'label': '🔴 CRITICAL',
        'description': 'Immediate risk — emergency response within 24 hours',
        'nerc_reference': 'FAC-003-4 R1/R2 imminent threat',
        'priority_hours': 24,
    },
    'high': {
        'label': '🟠 HIGH',
        'description': 'Significant risk — inspect within 30 days',
        'nerc_reference': 'FAC-003-4 R3 clearance violation',
        'priority_hours': 720,
    },
    'moderate': {
        'label': '🟡 MODERATE',
        'description': 'Monitor — address at next scheduled maintenance',
        'nerc_reference': 'FAC-003-4 preventive maintenance',
        'priority_hours': 2160,
    },
    'low': {
        'label': '🟢 LOW',
        'description': 'Informational — log for trend tracking',
        'nerc_reference': 'General condition monitoring',
        'priority_hours': 8760,
    },
}

# The JSON schema tells Pegasus exactly what format to respond in
INSPECTION_SCHEMA = {
    'type': 'object',
    'properties': {
        'asset_type': {'type': 'string'},
        'anomaly_detected': {'type': 'boolean'},
        'anomaly_type': {'type': 'string'},
        'condition_description': {'type': 'string'},
        'severity': {'type': 'string'},
        'confidence': {'type': 'number'},
        'recommended_action': {'type': 'string'}
    },
    'required': ['asset_type', 'anomaly_detected', 'anomaly_type',
                 'condition_description', 'severity', 'confidence', 'recommended_action']
}

print('✅ Severity framework and schema ready')

In [ ]:
# Process each finding with Pegasus
enriched_findings = []

for i, finding in enumerate(findings):
    fid = i + 1
    start = finding['start_time']
    end = finding['end_time']
    atype = finding['anomaly_type']
    
    print(f'\n[{fid}/{len(findings)}] Analyzing {atype.replace("_", " ")} at {start:.0f}s - {end:.0f}s ...')
    
    prompt = f"""You are a certified transmission line inspector.
Analyze the video segment from {start:.0f}s to {end:.0f}s.
This segment was flagged for potential {atype.replace('_', ' ')}.

Assess:
1. Insulator condition (cracked, chipped, contaminated, flashover marks)
2. Vegetation proximity to conductors
3. Tower/structure corrosion or damage

Rate severity as: critical (immediate danger), high (30-day fix), moderate (routine maintenance), or low (monitoring only)."""
    
    try:
        # Try structured output first
        request_body = {
            'inputPrompt': prompt,
            'mediaSource': {
                's3Location': {
                    'uri': video_s3_uri,
                    'bucketOwner': aws_account_id
                }
            },
            'temperature': 0,
            'responseFormat': {'jsonSchema': INSPECTION_SCHEMA}
        }
        
        response = bedrock_client.invoke_model(
            modelId=PEGASUS_MODEL_ID,
            body=json.dumps(request_body),
            contentType='application/json',
            accept='application/json'
        )
        
        response_body = json.loads(response.get('body').read())
        pegasus_data = json.loads(response_body['message'])
        print(f'  ✅ Pegasus: {pegasus_data.get("condition_description", "N/A")[:80]}')
        
    except Exception as e:
        print(f'  ⚠️ Structured output failed, trying streaming...')
        try:
            # Fallback: streaming without structured output
            request_body = {
                'inputPrompt': prompt,
                'mediaSource': {
                    's3Location': {
                        'uri': video_s3_uri,
                        'bucketOwner': aws_account_id
                    }
                },
                'temperature': 0
            }
            streaming_resp = bedrock_client.invoke_model_with_response_stream(
                modelId=PEGASUS_MODEL_ID,
                body=json.dumps(request_body),
                contentType='application/json',
                accept='application/json'
            )
            message = ''
            for event in streaming_resp['body']:
                chunk = json.loads(event['chunk']['bytes'])
                message += chunk.get('message', '')
            
            pegasus_data = {
                'asset_type': atype.split('_')[0],
                'anomaly_detected': True,
                'anomaly_type': atype,
                'condition_description': message[:500],
                'severity': 'moderate',
                'confidence': 0.5,
                'recommended_action': 'Field inspection recommended'
            }
            print(f'  ✅ Pegasus (streaming): {message[:80]}')
        except Exception as e2:
            print(f'  ❌ Pegasus failed: {e2}')
            pegasus_data = {
                'asset_type': 'unknown',
                'anomaly_detected': True,
                'anomaly_type': atype,
                'condition_description': f'Detected via search query: {finding["query"]}',
                'severity': 'moderate',
                'confidence': finding['score'],
                'recommended_action': 'Field inspection recommended'
            }
    
    # Apply NERC severity
    severity = pegasus_data.get('severity', 'moderate').lower()
    if severity not in SEVERITY_FRAMEWORK:
        severity = 'moderate'
    fw = SEVERITY_FRAMEWORK[severity]
    
    enriched = {
        'id': fid,
        'anomaly_type': atype,
        'start_time': start,
        'end_time': end,
        'marengo_score': finding['score'],
        'query_used': finding['query'],
        'asset_type': pegasus_data.get('asset_type', 'unknown'),
        'condition': pegasus_data.get('condition_description', ''),
        'severity': severity,
        'severity_label': fw['label'],
        'nerc_reference': fw['nerc_reference'],
        'priority_hours': fw['priority_hours'],
        'pegasus_confidence': pegasus_data.get('confidence', 0.5),
        'recommended_action': pegasus_data.get('recommended_action', ''),
    }
    enriched_findings.append(enriched)
    print(f'  Severity: {fw["label"]}')

print(f'\n{"═"*60}')
print(f'✅ All {len(enriched_findings)} findings enriched with Pegasus + NERC severity')

---
## STEP 5: Results & Visualizations

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

df = pd.DataFrame(enriched_findings)

# ── Summary ──
print('═' * 60)
print('⚡ GRIDGUARD — INSPECTION REPORT')
print('═' * 60)
print(f'Total findings: {len(df)}\n')
for sev in ['critical', 'high', 'moderate', 'low']:
    count = len(df[df['severity'] == sev])
    if count > 0:
        emoji = {'critical': '🔴', 'high': '🟠', 'moderate': '🟡', 'low': '🟢'}[sev]
        print(f'  {emoji} {sev.upper():10s}: {count} findings')

print(f'\nBy anomaly type:')
for atype in df['anomaly_type'].unique():
    count = len(df[df['anomaly_type'] == atype])
    print(f'  {atype.replace("_", " "):25s}: {count}')

In [ ]:
# Severity bar chart
sev_counts = df['severity'].value_counts().reindex(['critical', 'high', 'moderate', 'low'], fill_value=0)
fig1 = go.Figure(data=[go.Bar(
    x=['Critical', 'High', 'Moderate', 'Low'],
    y=sev_counts.values,
    marker_color=['#ef4444', '#f97316', '#eab308', '#22c55e']
)])
fig1.update_layout(title='Findings by Severity (NERC FAC-003)', height=350)
fig1.show()

In [ ]:
# Detection timeline
sev_colors = {'critical': '#ef4444', 'high': '#f97316', 'moderate': '#eab308', 'low': '#22c55e'}
fig2 = px.scatter(
    df, x='start_time', y='anomaly_type', color='severity',
    size='marengo_score', color_discrete_map=sev_colors,
    hover_data=['condition', 'recommended_action'],
    title='Detection Timeline — Where Anomalies Were Found',
    labels={'start_time': 'Video Time (seconds)', 'anomaly_type': ''}
)
fig2.update_layout(height=300)
fig2.show()

In [ ]:
# Full findings table sorted by severity
sev_order = {'critical': 0, 'high': 1, 'moderate': 2, 'low': 3}
df_display = df.copy()
df_display['sev_rank'] = df_display['severity'].map(sev_order)
df_display = df_display.sort_values('sev_rank')
df_display[['id', 'severity_label', 'anomaly_type', 'start_time', 'end_time',
            'condition', 'recommended_action', 'nerc_reference', 'marengo_score']]

---
## STEP 6: Export Results

In [ ]:
# Save CSV (for work order systems)
df.to_csv('gridguard_findings.csv', index=False)
print('✅ Saved: gridguard_findings.csv')

# Save JSON
with open('gridguard_findings.json', 'w') as f:
    json.dump(enriched_findings, f, indent=2)
print('✅ Saved: gridguard_findings.json')

# Upload to S3
s3_client.upload_file('gridguard_findings.csv', S3_BUCKET_NAME, 'output/gridguard_findings.csv')
s3_client.upload_file('gridguard_findings.json', S3_BUCKET_NAME, 'output/gridguard_findings.json')
print('✅ Uploaded results to S3')

---
## STEP 7: Validation Report (REQUIRED — 35% of your score)

**YOU MUST DO THIS MANUALLY:**
1. Watch your video from start to finish
2. Every time you see a real problem (damaged insulator, tree too close, rust), write down the start time, end time, and type
3. Put those labels in the GROUND_TRUTH list below
4. Run the cell — it compares your system's detections against your manual labels

Aim for 15-25 labels. This is what proves your system actually works.

In [ ]:
# ════════════════════════════════════════════════════════
# FILL THIS IN by watching your video
# Format: {'start': seconds, 'end': seconds, 'type': 'anomaly_type'}
# Types: 'insulator_damage', 'vegetation_encroachment', 'structural_corrosion'
# ════════════════════════════════════════════════════════
GROUND_TRUTH = [
    # Example (replace with your real observations):
    # {'start': 45, 'end': 52, 'type': 'insulator_damage'},
    # {'start': 120, 'end': 128, 'type': 'vegetation_encroachment'},
    # {'start': 195, 'end': 210, 'type': 'structural_corrosion'},
]

if not GROUND_TRUTH:
    print('⚠️  You need to fill in GROUND_TRUTH above!')
    print('Watch your video, note every anomaly, add it to the list.')
    print('Without this, you have NO validation metrics and judges will score you low.')
else:
    # Match detections to ground truth
    matched_gt = set()
    matched_det = set()
    for di, det in enumerate(enriched_findings):
        for gi, gt in enumerate(GROUND_TRUTH):
            if gi in matched_gt or det['anomaly_type'] != gt['type']:
                continue
            overlap = min(det['end_time'], gt['end']) - max(det['start_time'], gt['start'])
            if overlap >= 1.0:
                matched_gt.add(gi)
                matched_det.add(di)
                break
    
    tp = len(matched_det)
    fp = len(enriched_findings) - tp
    fn = len(GROUND_TRUTH) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print('═' * 50)
    print('⚡ GRIDGUARD VALIDATION REPORT')
    print('═' * 50)
    print(f'Ground truth labels:  {len(GROUND_TRUTH)}')
    print(f'System detections:    {len(enriched_findings)}')
    print(f'True positives:       {tp}')
    print(f'False positives:      {fp}')
    print(f'False negatives:      {fn}')
    print(f'')
    print(f'Precision:  {precision:.1%}')
    print(f'Recall:     {recall:.1%}')
    print(f'F1 Score:   {f1:.1%}')
    print(f'')
    print(f'Methodology:')
    print(f'  Detection: Marengo 3.0 semantic search ({sum(len(v) for v in DETECTION_QUERIES.values())} queries)')
    print(f'  Description: Pegasus 1.2 structured output')
    print(f'  Severity: NERC FAC-003-4 / FAC-501-3 thresholds')
    print(f'  Matching: Temporal overlap ≥ 1s with type match')

---
## STEP 8: Video Evidence Player

Play the video at specific timestamps to verify findings.

In [ ]:
# Generate presigned URL to play the video from S3
presigned_url = s3_client.generate_presigned_url(
    'get_object',
    Params={'Bucket': S3_BUCKET_NAME, 'Key': video_s3_key},
    ExpiresIn=3600
)

def play_at(start_time):
    """Play the video starting at a specific time."""
    html = f'<video width="640" controls><source src="{presigned_url}#t={start_time}" type="video/mp4"></video>'
    display(HTML(html))

# Show evidence for top findings
print('Click play to verify each finding:\n')
for f in sorted(enriched_findings, key=lambda x: {'critical':0,'high':1,'moderate':2,'low':3}.get(x['severity'],4))[:5]:
    print(f'{f["severity_label"]} #{f["id"]} — {f["anomaly_type"].replace("_"," ")} at {f["start_time"]:.0f}s')
    print(f'  {f["condition"][:100]}')
    play_at(f['start_time'])